In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
import wfdb
import json
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, average_precision_score, confusion_matrix, roc_curve, precision_recall_curve, auc

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_PATH = r"e:\平时作业\人工智能\data\master_index.csv"

In [ ]:
# Re-define classes to load the saved state_dicts from Stage 1

class VisualEngine(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = models.resnet18(pretrained=False) # Load structure
        self.feature_extractor = nn.Sequential(*list(self.model.children())[:-1])
        # We only need `feature_extractor` part from the saved file potentially, 
        # but if we saved the whole `VisualEngine`, we need to match that structure.
        # In Stage 1, VisualEngine had `feature_extractor` and `classifier`.
        self.classifier = nn.Sequential(nn.Linear(512, 128), nn.ReLU(), nn.Linear(128, 1))

    def forward(self, x):
        features = self.feature_extractor(x)
        features = features.view(features.size(0), -1)
        return features # Return features for fusion

class ResBlock1D(nn.Module): # Dependency for ECG
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size=7, stride=stride, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size=7, stride=1, padding=3, bias=False)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_ch)
            )
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return self.relu(out)

class ECGResNet1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        )
        self.layer1 = ResBlock1D(32, 64, stride=2)
        self.layer2 = ResBlock1D(64, 128, stride=2)
        self.layer3 = ResBlock1D(128, 256, stride=2)
        self.layer4 = ResBlock1D(256, 512, stride=2)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(512, 1)
        self.decoder = nn.Sequential(nn.Linear(512, 1024), nn.ReLU(), nn.Linear(1024, 2000))

    def forward(self, x):
        features = self.stem(x)
        features = self.layer1(features)
        features = self.layer2(features)
        features = self.layer3(features)
        features = self.layer4(features)
        latent = self.global_pool(features).squeeze(-1)
        return latent # Return features

class TextEngine(nn.Module):
    def __init__(self, model_name="emilyalsentzer/Bio_ClinicalBERT"):
        super().__init__()
        try:
            self.bert = AutoModel.from_pretrained(model_name)
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        except:
            self.bert = AutoModel.from_pretrained("distilbert-base-uncased")
            self.tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
        self.fc = nn.Linear(768, 512)
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_token = outputs.last_hidden_state[:, 0, :]
        return self.fc(cls_token)

In [ ]:
# ==========================================
# Agent Logic: Time-Aware Attention Injection
# ==========================================
def agent_analyze_time_urgency(text_batch):
    """
    Simulates AI Agent analysis of clinical notes for temporal keywords.
    Returns: attention_bias [B, 3] (Text, ECG, Image)
    """
    bias_list = []
    acute_kw = ['acute', 'sudden', 'onset', 'emergency', 'worsening', 'decompensation', 'flash']
    chronic_kw = ['chronic', 'history', 'compliance', 'medication', 'social', 'discharge', 'follow-up']
    
    for text in text_batch:
        txt_lower = text.lower()
        score_acute = sum(1 for w in acute_kw if w in txt_lower)
        score_chronic = sum(1 for w in chronic_kw if w in txt_lower)
        bias = [0.0, 0.0, 0.0]
        if score_acute > score_chronic:
            bias = [-0.5, 1.0, 1.0]
        elif score_chronic > score_acute:
            bias = [1.0, -0.5, -0.5]
        bias_list.append(bias)
    return torch.tensor(bias_list, dtype=torch.float32).to(DEVICE)

# ==========================================
# Fusion Model v3: Softmax Masking + Uncertainty Weighting
# ==========================================
class AgentDrivenFusion(nn.Module):
    def __init__(self, 
                 drop_text_prob=0.30,
                 drop_ecg_prob=0.10,
                 drop_img_prob=0.30):
        super().__init__()
        self.text_eng = TextEngine()
        self.ecg_eng = ECGResNet1D()
        self.vis_eng = VisualEngine()
        
        # Modality Dropout Probabilities
        self.drop_text_prob = drop_text_prob
        self.drop_ecg_prob = drop_ecg_prob
        self.drop_img_prob = drop_img_prob
        
        # Load Pre-trained Weights if available
        if os.path.exists("text_encoder.pth"):
            self.text_eng.load_state_dict(torch.load("text_encoder.pth", map_location=DEVICE), strict=False)
            print("  ✅ Loaded text_encoder.pth")
        if os.path.exists("ecg_encoder.pth"):
            self.ecg_eng.load_state_dict(torch.load("ecg_encoder.pth", map_location=DEVICE), strict=False)
            print("  ✅ Loaded ecg_encoder.pth")
        if os.path.exists("visual_encoder.pth"):
            self.vis_eng.load_state_dict(torch.load("visual_encoder.pth", map_location=DEVICE), strict=False)
            print("  ✅ Loaded visual_encoder.pth")
            
        # Fusion Gating Layer
        self.gate_net = nn.Sequential(
            nn.Linear(512 * 3, 128),
            nn.ReLU(),
            nn.Linear(128, 3)
        )
        
        # Main Classifier (fused features -> prediction)
        self.classifier = nn.Sequential(
            nn.Linear(512 * 3, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 1)
        )
        
        # ★ ECG Auxiliary Classifier (ecg features alone -> prediction)
        self.ecg_aux_classifier = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )
        
        # ★ NEW v3: Uncertainty Weighting — learnable log(σ²) for automatic loss balancing
        # loss_params[0] = log(σ²) for main loss
        # loss_params[1] = log(σ²) for ECG aux loss
        # Formula: L = (1/(2σ²)) * L_task + (1/2)*log(σ²)
        self.loss_params = nn.Parameter(torch.zeros(2))  # initialized to log(σ²)=0 → σ²=1
        
        # Running mean vectors for each modality (for ablation replacement)
        self.register_buffer('text_mean', torch.zeros(512))
        self.register_buffer('ecg_mean', torch.zeros(512))
        self.register_buffer('img_mean', torch.zeros(512))
        self.register_buffer('mean_initialized', torch.tensor(False))
        self._ema_decay = 0.99
        
    def _update_running_means(self, f_text, f_ecg, f_img):
        """Exponential Moving Average of feature means (for ablation use)"""
        with torch.no_grad():
            batch_text_mean = f_text.detach().mean(dim=0)
            batch_ecg_mean = f_ecg.detach().mean(dim=0)
            batch_img_mean = f_img.detach().mean(dim=0)
            
            if not self.mean_initialized:
                self.text_mean.copy_(batch_text_mean)
                self.ecg_mean.copy_(batch_ecg_mean)
                self.img_mean.copy_(batch_img_mean)
                self.mean_initialized.fill_(True)
            else:
                d = self._ema_decay
                self.text_mean.mul_(d).add_(batch_text_mean, alpha=1 - d)
                self.ecg_mean.mul_(d).add_(batch_ecg_mean, alpha=1 - d)
                self.img_mean.mul_(d).add_(batch_img_mean, alpha=1 - d)
    
    def compute_loss(self, loss_main, loss_ecg_aux):
        """
        ★ v3: Uncertainty Weighting — automatic loss balancing via learnable σ.
        L = (1/(2*σ₁²)) * L_main + (1/2)*log(σ₁²)
          + (1/(2*σ₂²)) * L_ecg  + (1/2)*log(σ₂²)
        The log(σ²) regularization prevents σ → ∞ (ignoring a loss).
        """
        precision_main = torch.exp(-self.loss_params[0])   # 1/σ₁²
        precision_ecg  = torch.exp(-self.loss_params[1])    # 1/σ₂²
        
        loss = 0.5 * precision_main * loss_main + 0.5 * self.loss_params[0] \
             + 0.5 * precision_ecg  * loss_ecg_aux + 0.5 * self.loss_params[1]
        return loss
        
    def forward(self, ecg, img, input_ids, mask, agent_bias):
        # 1. Feature Extraction
        f_text = self.text_eng(input_ids, mask)   # [B, 512]
        f_ecg = self.ecg_eng(ecg)                 # [B, 512]
        f_img = self.vis_eng(img)                  # [B, 512]
        
        # ★ ECG Auxiliary Logits (before any dropout — pure ECG signal)
        ecg_aux_logits = self.ecg_aux_classifier(f_ecg)  # [B, 1]
        
        # ★ Update running means during training
        if self.training:
            self._update_running_means(f_text, f_ecg, f_img)
        
        B = f_text.size(0)
        
        # ★ v3: MODALITY DROPOUT with proper Softmax MASKING
        # drop_mask[i] = True means modality i is dropped
        drop_mask = torch.zeros(B, 3, device=f_text.device, dtype=torch.bool)
        
        if self.training:
            if torch.rand(1).item() < self.drop_text_prob:
                f_text = torch.zeros_like(f_text)
                drop_mask[:, 0] = True
            if torch.rand(1).item() < self.drop_ecg_prob:
                f_ecg_fused = torch.zeros_like(f_ecg)
                drop_mask[:, 1] = True
            else:
                f_ecg_fused = f_ecg
            if torch.rand(1).item() < self.drop_img_prob:
                f_img = torch.zeros_like(f_img)
                drop_mask[:, 2] = True
            f_ecg_for_gate = f_ecg_fused
        else:
            f_ecg_for_gate = f_ecg
        
        # 2. Agent-Informed Gating
        concat_raw = torch.cat([f_text, f_ecg_for_gate, f_img], dim=1)
        gate_logits = self.gate_net(concat_raw)
        prior_driven_logits = gate_logits + agent_bias
        
        # ★★★ v3 CRITICAL FIX: Mask dropped modalities BEFORE Softmax
        # Setting logits to -inf ensures exp(-inf)=0 → Softmax gives 0 weight
        # This prevents zeroed features from stealing attention via e^0=1
        if self.training:
            # Safety: ensure at least one modality is NOT masked
            all_dropped = drop_mask.all(dim=1)  # [B] — True if ALL 3 dropped
            if all_dropped.any():
                # Extremely rare: undo all drops (keep all modalities)
                drop_mask[all_dropped] = False
            prior_driven_logits = prior_driven_logits.masked_fill(drop_mask, -1e9)
        
        weights = F.softmax(prior_driven_logits, dim=1)  # [B, 3]
        
        # 3. Weighted Fusion
        f_text_w = f_text * weights[:, 0:1]
        f_ecg_w = f_ecg_for_gate * weights[:, 1:2]
        f_img_w = f_img * weights[:, 2:3]
        
        fused = torch.cat([f_text_w, f_ecg_w, f_img_w], dim=1)
        
        # 4. Main Classification
        logits = self.classifier(fused)
        
        return logits, weights, ecg_aux_logits

print("Agent-Driven Fusion Model Initialized (v3).")
print("  ★ Softmax Masking: Dropped modalities get -inf logits → 0 attention weight")
print("  ★ Uncertainty Weighting: Learnable σ auto-balances Main vs ECG Aux loss")
print("  ★ Modality Dropout: Text=30%, ECG=10%, Image=30%")
print("  ★ ECG Auxiliary Loss Branch + Running Mean Vectors")

In [ ]:
# [SETUP] Dataset, DataLoaders, and Model Initialization
# Run this cell FIRST to prepare environment for Training or Evaluation

class FusionDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.df = df
        self.tokenizer = tokenizer
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Text
        text_raw = "No data"
        if pd.notna(row['text_path']) and os.path.exists(row['text_path']):
            with open(row['text_path'], 'r', encoding='utf-8') as f: text_raw = f.read()[:1000]
        
        tokens = self.tokenizer(text_raw, max_length=128, padding='max_length', truncation=True, return_tensors='pt')
        
        # ECG
        ecg = np.zeros(2000, dtype=np.float32)
        if pd.notna(row['ecg_path']) and os.path.exists(row['ecg_path']):
            try:
                sig, _ = wfdb.rdsamp(row['ecg_path'], channels=[0])
                raw = sig.flatten()
                if len(raw) >= 2000: ecg = raw[:2000]
                else: ecg[:len(raw)] = raw
            except: pass
            
        # Image
        img_tens = torch.zeros(3, 224, 224)
        if pd.notna(row['image_path']) and os.path.exists(row['image_path']):
            try:
                img_o = Image.open(row['image_path']).convert('RGB')
                img_tens = self.transform(img_o)
            except: pass
            
        return {
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
            'ecg': torch.tensor(ecg, dtype=torch.float32).unsqueeze(0),
            'image': img_tens,
            'label': torch.tensor(float(row['label']), dtype=torch.float32),
            'raw_text': text_raw # For Agent Analysis
        }

# Setup
print("Initializing Data and Model...")
df_master = pd.read_csv(DATA_PATH).sample(frac=1.0, random_state=42) # Load Master
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")

# Split Train/Test (80/20)
train_size = int(0.8 * len(df_master))
df_train = df_master.iloc[:train_size].reset_index(drop=True)
df_test = df_master.iloc[train_size:].reset_index(drop=True)

train_dataset = FusionDataset(df_train, tokenizer)
test_dataset = FusionDataset(df_test, tokenizer)

# Define Loaders (Global scope for access in other cells)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Model Init
model = AgentDrivenFusion().to(DEVICE)
criterion = nn.BCEWithLogitsLoss()

# Metrics History
history = {
    'train_loss': [], 'val_loss': [], 
    'val_auc': [], 'val_f1': [], 'val_ap': [], 
    'gate_weights': [] 
}

print(f"Setup Complete. Train Size: {len(train_dataset)}, Test Size: {len(test_dataset)}")
print("Variables ready: train_loader, test_loader, model, history")

In [ ]:
# [TRAINING] v3: Softmax Masking + Uncertainty Weighting + Dynamic Threshold
# ★ Fixes: F1=0 caused by Softmax dilution of dropped modalities

def find_best_threshold(targets, probs):
    """
    ★ v3: Dynamic threshold search via Precision-Recall curve.
    Instead of fixed 0.5, find threshold that maximizes F1 on the validation set.
    Critical for imbalanced data where positive probabilities cluster below 0.5.
    """
    precisions, recalls, thresholds = precision_recall_curve(targets, probs)
    # F1 = 2*P*R / (P+R)
    f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-8)
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    best_f1 = f1_scores[best_idx]
    return best_threshold, best_f1


def evaluate(model, loader, criterion):
    """Evaluate model — reports both fixed-0.5 and best-threshold F1."""
    model.eval()
    val_loss = 0
    all_preds = []
    all_targets = []
    all_ecg_preds = []
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(DEVICE)
            att_mask = batch['attention_mask'].to(DEVICE)
            ecg = batch['ecg'].to(DEVICE)
            img = batch['image'].to(DEVICE)
            labels = batch['label'].to(DEVICE).unsqueeze(1)
            agent_bias = agent_analyze_time_urgency(batch['raw_text'])
            
            logits, weights, ecg_aux = model(ecg, img, input_ids, att_mask, agent_bias)
            loss = criterion(logits, labels)
            val_loss += loss.item()
            
            probs = torch.sigmoid(logits).cpu().numpy()
            ecg_probs = torch.sigmoid(ecg_aux).cpu().numpy()
            all_preds.extend(probs)
            all_targets.extend(labels.cpu().numpy())
            all_ecg_preds.extend(ecg_probs)
            
    val_loss /= len(loader)
    all_preds = np.array(all_preds).flatten()
    all_targets = np.array(all_targets).flatten()
    all_ecg_preds = np.array(all_ecg_preds).flatten()
    
    try:
        auc_val = roc_auc_score(all_targets, all_preds)
        ap_val = average_precision_score(all_targets, all_preds)
        
        # ★ v3: Fixed threshold F1
        preds_bin_fixed = (all_preds > 0.5).astype(int)
        f1_fixed = f1_score(all_targets, preds_bin_fixed, zero_division=0)
        
        # ★ v3: Dynamic best-threshold F1
        best_thr, f1_best = find_best_threshold(all_targets, all_preds)
        
    except:
        auc_val = 0.5; ap_val = 0; f1_fixed = 0; f1_best = 0; best_thr = 0.5
    
    # ECG branch standalone AUC
    try:
        ecg_auc = roc_auc_score(all_targets, all_ecg_preds)
    except:
        ecg_auc = 0.5

    return val_loss, auc_val, f1_fixed, f1_best, best_thr, ap_val, ecg_auc


def train_one_epoch(model, loader, optimizer, criterion, epoch_label):
    """Train one epoch with Softmax Masking + Uncertainty Weighting."""
    model.train()
    total_loss = 0
    total_main = 0
    total_ecg_aux = 0
    epoch_gates = []
    
    for batch in tqdm(loader, desc=epoch_label):
        input_ids = batch['input_ids'].to(DEVICE)
        att_mask = batch['attention_mask'].to(DEVICE)
        ecg = batch['ecg'].to(DEVICE)
        img = batch['image'].to(DEVICE)
        labels = batch['label'].to(DEVICE).unsqueeze(1)
        agent_bias = agent_analyze_time_urgency(batch['raw_text'])
        
        optimizer.zero_grad()
        
        # Forward (Modality Dropout + Softmax Masking automatic in training mode)
        logits, weights, ecg_aux_logits = model(ecg, img, input_ids, att_mask, agent_bias)
        
        # ★ Individual losses
        loss_main = criterion(logits, labels)
        loss_ecg_aux = criterion(ecg_aux_logits, labels)
        
        # ★ v3: Uncertainty Weighting — model auto-balances losses via learnable σ
        loss = model.compute_loss(loss_main, loss_ecg_aux)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_main += loss_main.item()
        total_ecg_aux += loss_ecg_aux.item()
        epoch_gates.append(weights.mean(dim=0).detach().cpu().numpy())
    
    n = len(loader)
    return total_loss / n, total_main / n, total_ecg_aux / n, epoch_gates


# ==========================================
# STAGE 2-A: FREEZE ENCODERS, TRAIN FUSION + GATE + ECG_AUX
# ==========================================
print("=" * 60)
print("❄ STAGE 2-A: Freezing Expert Encoders (v3)")
print(f"  ★ Softmax Masking: dropped modalities → -inf logits → 0 weight")
print(f"  ★ Uncertainty Weighting: auto-balances Main vs ECG Aux loss")
print(f"  ★ Dynamic Threshold: auto-searches best F1 threshold per epoch")
print(f"  ★ Modality Dropout: Text=30%, ECG=10%, Image=30%")
print("=" * 60)

for param in model.text_eng.parameters(): param.requires_grad = False
for param in model.ecg_eng.parameters(): param.requires_grad = False
for param in model.vis_eng.parameters(): param.requires_grad = False

# ★ v3: Include loss_params in optimizer (they are NOT part of the frozen encoders)
trainable_params = filter(lambda p: p.requires_grad, model.parameters())
optimizer = optim.Adam(trainable_params, lr=1e-3)

for epoch in range(3):
    avg_loss, avg_main, avg_ecg, epoch_gates = train_one_epoch(
        model, train_loader, optimizer, criterion, f"Stage 2A Epoch {epoch+1}"
    )
    v_loss, v_auc, v_f1_fixed, v_f1_best, v_thr, v_ap, v_ecg_auc = evaluate(model, test_loader, criterion)
    
    # ★ v3: Show learned loss weights (σ values)
    sigma_main = torch.exp(0.5 * model.loss_params[0]).item()
    sigma_ecg = torch.exp(0.5 * model.loss_params[1]).item()
    w_main = 1.0 / (2.0 * sigma_main**2)
    w_ecg  = 1.0 / (2.0 * sigma_ecg**2)
    
    print(f"  Loss: {avg_loss:.4f} (Main={avg_main:.4f}, ECG_Aux={avg_ecg:.4f})")
    print(f"  Learned weights: w_main={w_main:.3f}, w_ecg={w_ecg:.3f} (σ_main={sigma_main:.3f}, σ_ecg={sigma_ecg:.3f})")
    print(f"  Val: AUC={v_auc:.3f} | F1@0.5={v_f1_fixed:.3f} | F1@best={v_f1_best:.3f} (thr={v_thr:.3f}) | ECG-only AUC={v_ecg_auc:.3f}")
    
    history['train_loss'].append(avg_loss)
    history['val_loss'].append(v_loss)
    history['val_auc'].append(v_auc)
    history['val_f1'].append(v_f1_best)  # Use best-threshold F1 for tracking
    history['val_ap'].append(v_ap)
    history['gate_weights'].append(np.mean(epoch_gates, axis=0).tolist())

# ==========================================
# STAGE 2-B: UNFREEZE ALL FOR GLOBAL FINE-TUNING
# ==========================================
print("\n" + "=" * 60)
print("🔥 STAGE 2-B: Unfreezing All Parameters (v3)")
print("=" * 60)

for param in model.parameters(): param.requires_grad = True
optimizer = optim.Adam(model.parameters(), lr=5e-5)  # ★ v3: raised from 1e-5 to 5e-5

best_val_auc = max(history['val_auc']) if history['val_auc'] else 0
best_threshold = 0.5  # Will be updated each epoch

for epoch in range(5):
    avg_loss, avg_main, avg_ecg, epoch_gates = train_one_epoch(
        model, train_loader, optimizer, criterion, f"Stage 2B Epoch {epoch+1}"
    )
    v_loss, v_auc, v_f1_fixed, v_f1_best, v_thr, v_ap, v_ecg_auc = evaluate(model, test_loader, criterion)
    
    # ★ v3: Learned loss weights
    sigma_main = torch.exp(0.5 * model.loss_params[0]).item()
    sigma_ecg = torch.exp(0.5 * model.loss_params[1]).item()
    w_main = 1.0 / (2.0 * sigma_main**2)
    w_ecg  = 1.0 / (2.0 * sigma_ecg**2)
    
    print(f"  Loss: {avg_loss:.4f} (Main={avg_main:.4f}, ECG_Aux={avg_ecg:.4f})")
    print(f"  Learned weights: w_main={w_main:.3f}, w_ecg={w_ecg:.3f} (σ_main={sigma_main:.3f}, σ_ecg={sigma_ecg:.3f})")
    print(f"  Val: AUC={v_auc:.3f} | F1@0.5={v_f1_fixed:.3f} | F1@best={v_f1_best:.3f} (thr={v_thr:.3f}) | ECG-only AUC={v_ecg_auc:.3f}")
    
    history['train_loss'].append(avg_loss)
    history['val_loss'].append(v_loss)
    history['val_auc'].append(v_auc)
    history['val_f1'].append(v_f1_best)
    history['val_ap'].append(v_ap)
    history['gate_weights'].append(np.mean(epoch_gates, axis=0).tolist())
    
    # Save best model
    if v_auc > best_val_auc:
        best_val_auc = v_auc
        best_threshold = v_thr
        torch.save(model.state_dict(), "fusion_model_final.pth")
        print(f"  💾 New best model! AUC={v_auc:.3f}, best_thr={v_thr:.3f}")

print(f"\n✅ Training Complete. Best Val AUC: {best_val_auc:.3f}")
print(f"   Best Threshold (for inference): {best_threshold:.3f}")
print("   Model saved: fusion_model_final.pth")

In [ ]:
# ====================================
# Cell: Ablation Study (v3 — Dynamic Threshold + Mean Vector Replacement)
# ====================================
# ★ v3: Uses best_threshold from training instead of fixed 0.5

print("\n" + "="*60)
print("🔬 Ablation Study: Modality Contribution Analysis (v3)")
print("="*60)

# Use the best threshold from training, fallback to 0.5
try:
    abl_threshold = best_threshold
except NameError:
    abl_threshold = 0.5
print(f"   Using threshold: {abl_threshold:.3f}")

# Check if mean vectors are available
if hasattr(model, 'mean_initialized') and model.mean_initialized.item():
    print("✅ Using training-set Mean Vectors for ablation masking")
    print(f"   text_mean norm: {model.text_mean.norm():.4f}")
    print(f"   ecg_mean norm:  {model.ecg_mean.norm():.4f}")
    print(f"   img_mean norm:  {model.img_mean.norm():.4f}")
else:
    print("⚠️  Mean vectors not initialized. Computing from test set...")
    model.eval()
    all_text_feats = []
    all_ecg_feats = []
    all_img_feats = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            att_mask = batch['attention_mask'].to(DEVICE)
            ecg = batch['ecg'].to(DEVICE)
            img = batch['image'].to(DEVICE)
            f_text = model.text_eng(input_ids, att_mask)
            f_ecg = model.ecg_eng(ecg)
            f_img = model.vis_eng(img)
            all_text_feats.append(f_text)
            all_ecg_feats.append(f_ecg)
            all_img_feats.append(f_img)
    model.text_mean = torch.cat(all_text_feats).mean(dim=0)
    model.ecg_mean = torch.cat(all_ecg_feats).mean(dim=0)
    model.img_mean = torch.cat(all_img_feats).mean(dim=0)
    print("   ✅ Computed mean vectors from test set")

def evaluate_ablation_v3(model, loader, mask_modality=None, threshold=0.5):
    """
    Ablation evaluation using MEAN VECTOR REPLACEMENT + Dynamic Threshold.
    ★ v3: Uses learned best_threshold instead of fixed 0.5
    """
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(DEVICE)
            att_mask = batch['attention_mask'].to(DEVICE)
            ecg = batch['ecg'].to(DEVICE)
            img = batch['image'].to(DEVICE)
            labels = batch['label'].to(DEVICE).unsqueeze(1)
            agent_bias = agent_analyze_time_urgency(batch['raw_text'])
            
            # Extract features normally
            f_text = model.text_eng(input_ids, att_mask)
            f_ecg = model.ecg_eng(ecg)
            f_img = model.vis_eng(img)
            
            B = f_text.size(0)
            
            # ★ MEAN VECTOR REPLACEMENT (not zeros!)
            if mask_modality == 'text':
                f_text = model.text_mean.unsqueeze(0).expand(B, -1)
            elif mask_modality == 'ecg':
                f_ecg = model.ecg_mean.unsqueeze(0).expand(B, -1)
            elif mask_modality == 'image':
                f_img = model.img_mean.unsqueeze(0).expand(B, -1)
            
            # Run fusion manually (bypass model.forward to avoid modality dropout)
            concat_raw = torch.cat([f_text, f_ecg, f_img], dim=1)
            gate_logits = model.gate_net(concat_raw)
            prior_driven_logits = gate_logits + agent_bias
            weights = F.softmax(prior_driven_logits, dim=1)
            
            f_text_w = f_text * weights[:, 0:1]
            f_ecg_w = f_ecg * weights[:, 1:2]
            f_img_w = f_img * weights[:, 2:3]
            fused = torch.cat([f_text_w, f_ecg_w, f_img_w], dim=1)
            
            logits = model.classifier(fused)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_preds.extend(probs)
            all_targets.extend(labels.cpu().numpy())
            
    all_preds = np.array(all_preds).flatten()
    all_targets = np.array(all_targets).flatten()
    
    try:
        auc_val = roc_auc_score(all_targets, all_preds)
        # ★ v3: Use dynamic threshold
        preds_bin = (all_preds > threshold).astype(int)
        f1 = f1_score(all_targets, preds_bin, zero_division=0)
        rec = recall_score(all_targets, preds_bin, zero_division=0)
        prec = precision_score(all_targets, preds_bin, zero_division=0)
    except:
        auc_val = 0.5; f1 = 0; rec = 0; prec = 0
    
    return auc_val, f1, rec, prec

# Run Ablations
print(f"\nRunning ablation experiments (threshold={abl_threshold:.3f})...")
baseline_auc, baseline_f1, baseline_rec, baseline_prec = evaluate_ablation_v3(model, test_loader, mask_modality=None, threshold=abl_threshold)
no_text_auc, no_text_f1, no_text_rec, no_text_prec = evaluate_ablation_v3(model, test_loader, mask_modality='text', threshold=abl_threshold)
no_ecg_auc, no_ecg_f1, no_ecg_rec, no_ecg_prec = evaluate_ablation_v3(model, test_loader, mask_modality='ecg', threshold=abl_threshold)
no_img_auc, no_img_f1, no_img_rec, no_img_prec = evaluate_ablation_v3(model, test_loader, mask_modality='image', threshold=abl_threshold)

print(f"\n{'Configuration':<25} | {'AUC':<8} | {'F1':<8} | {'Prec':<8} | {'Recall':<8} | {'Drop(AUC)':<10}")
print("-" * 80)
print(f"{'Full Model':<25} | {baseline_auc:.4f}   | {baseline_f1:.4f}   | {baseline_prec:.4f}   | {baseline_rec:.4f}   | -")
print(f"{'No Text (→ mean)':<25} | {no_text_auc:.4f}   | {no_text_f1:.4f}   | {no_text_prec:.4f}   | {no_text_rec:.4f}   | {baseline_auc - no_text_auc:+.4f}")
print(f"{'No ECG (→ mean)':<25} | {no_ecg_auc:.4f}   | {no_ecg_f1:.4f}   | {no_ecg_prec:.4f}   | {no_ecg_rec:.4f}   | {baseline_auc - no_ecg_auc:+.4f}")
print(f"{'No Image (→ mean)':<25} | {no_img_auc:.4f}   | {no_img_f1:.4f}   | {no_img_prec:.4f}   | {no_img_rec:.4f}   | {baseline_auc - no_img_auc:+.4f}")

# ECG Auxiliary Branch standalone
print("\n--- ECG Auxiliary Branch (Standalone) ---")
model.eval()
ecg_only_preds = []
ecg_only_targets = []
with torch.no_grad():
    for batch in test_loader:
        ecg = batch['ecg'].to(DEVICE)
        labels = batch['label'].numpy()
        f_ecg = model.ecg_eng(ecg)
        ecg_logits = model.ecg_aux_classifier(f_ecg)
        ecg_only_preds.extend(torch.sigmoid(ecg_logits).cpu().numpy().flatten())
        ecg_only_targets.extend(labels)
try:
    ecg_standalone_auc = roc_auc_score(ecg_only_targets, ecg_only_preds)
    ecg_thr, ecg_f1 = find_best_threshold(np.array(ecg_only_targets), np.array(ecg_only_preds))
    print(f"ECG-Only AUC (aux branch): {ecg_standalone_auc:.4f}")
    print(f"ECG-Only Best F1: {ecg_f1:.4f} (threshold={ecg_thr:.3f})")
except:
    print("ECG-Only AUC: N/A")

# Plot
drops = [baseline_auc - no_text_auc, baseline_auc - no_ecg_auc, baseline_auc - no_img_auc]
modalities = ['Text', 'ECG', 'Image']
colors = ['#2196F3' if d > 0 else '#F44336' for d in drops]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of AUC drops
bars = axes[0].bar(modalities, drops, color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('AUC Drop on Modality Removal\n(Higher = More Important)', fontsize=13)
axes[0].set_ylabel('AUC Drop', fontsize=12)
axes[0].axhline(y=0, color='black', linewidth=0.5)
axes[0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars, drops):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, 
                f'{val:+.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Comparison bar chart
x = np.arange(4)
aucs = [baseline_auc, no_text_auc, no_ecg_auc, no_img_auc]
labels_bar = ['Full', 'No Text', 'No ECG', 'No Image']
bar_colors = ['#4CAF50', '#2196F3', '#FF9800', '#F44336']
axes[1].bar(x, aucs, color=bar_colors, edgecolor='black', linewidth=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels_bar)
axes[1].set_title(f'AUC by Configuration\n(Mean Vector Replacement, thr={abl_threshold:.3f})', fontsize=13)
axes[1].set_ylabel('AUC', fontsize=12)
axes[1].set_ylim(0, 1.0)
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(aucs):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ====================================
# Cell: Comprehensive Visualization (v2)
# ====================================
print("\n" + "="*60)
print("📊 Comprehensive Evaluation Results")
print("="*60)

# Generate predictions using current model
print("Generating predictions (Inference on Test Set)...")
model.eval()
y_true = []
y_pred_prob = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluation"):
        input_ids = batch['input_ids'].to(DEVICE)
        att_mask = batch['attention_mask'].to(DEVICE)
        ecg = batch['ecg'].to(DEVICE)
        img = batch['image'].to(DEVICE)
        labels = batch['label'].to(DEVICE).unsqueeze(1)
        agent_bias = agent_analyze_time_urgency(batch['raw_text'])
        
        logits, _, _ = model(ecg, img, input_ids, att_mask, agent_bias)
        probs = torch.sigmoid(logits).cpu().numpy()
        
        y_pred_prob.extend(probs)
        y_true.extend(labels.cpu().numpy())

y_true = np.array(y_true).flatten()
y_pred_prob = np.array(y_pred_prob).flatten()

# Calculate metrics
try:
    cur_auc = roc_auc_score(y_true, y_pred_prob)
    cur_ap = average_precision_score(y_true, y_pred_prob)
    y_pred_bin = (y_pred_prob > 0.5).astype(int)
    cur_f1 = f1_score(y_true, y_pred_bin, zero_division=0)
    cur_prec = precision_score(y_true, y_pred_bin, zero_division=0)
    cur_rec = recall_score(y_true, y_pred_bin, zero_division=0)
except:
    cur_auc=0.5; cur_ap=0; cur_f1=0; cur_prec=0; cur_rec=0; y_pred_bin=(y_pred_prob>0.5).astype(int)

print(f"Count: {len(y_true)} | AUC: {cur_auc:.4f} | F1: {cur_f1:.4f} | AP: {cur_ap:.4f}")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_pred_prob)
axes[0, 0].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {cur_auc:.2f})')
axes[0, 0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[0, 0].set_xlim([0.0, 1.0]); axes[0, 0].set_ylim([0.0, 1.05])
axes[0, 0].set_xlabel('False Positive Rate'); axes[0, 0].set_ylabel('True Positive Rate')
axes[0, 0].set_title('ROC Curve'); axes[0, 0].legend(loc="lower right")
axes[0, 0].grid(True, alpha=0.3)

# 2. Confusion Matrix
cm = confusion_matrix(y_true, y_pred_bin)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 1])
axes[0, 1].set_title('Confusion Matrix')
axes[0, 1].set_xlabel('Predicted'); axes[0, 1].set_ylabel('True')

# 3. Training Curves
if len(history.get('train_loss', [])) > 0:
    epochs_range = range(1, len(history['train_loss']) + 1)
    axes[1, 0].plot(epochs_range, history['train_loss'], 'b-', label='Train Loss', marker='.')
    
    val_auc_hist = history.get('val_auc', [])
    if len(val_auc_hist) > 0:
        axes[1, 0].plot(range(1, len(val_auc_hist)+1), val_auc_hist, 'g--', label='Val AUC', marker='o')
    
    val_f1_hist = history.get('val_f1', [])
    if len(val_f1_hist) > 0:
        axes[1, 0].plot(range(1, len(val_f1_hist)+1), val_f1_hist, 'r-.', label='Val F1', marker='s')
    
    axes[1, 0].axvline(x=3.5, color='k', linestyle=':', label='Stage A→B', alpha=0.7)
    axes[1, 0].set_title('Training Metrics'); axes[1, 0].set_xlabel('Epochs')
    axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)
else:
    axes[1, 0].text(0.5, 0.5, 'No Training History', ha='center', va='center', 
                   transform=axes[1, 0].transAxes, fontsize=14)

# 4. Attention Weights Evolution
if len(history.get('gate_weights', [])) > 0:
    gate_weights = np.array(history['gate_weights'])
    gw_range = range(1, len(gate_weights) + 1)
    axes[1, 1].plot(gw_range, gate_weights[:, 0], label='Text', marker='o', color='#2196F3')
    axes[1, 1].plot(gw_range, gate_weights[:, 1], label='ECG', marker='s', color='#FF9800')
    axes[1, 1].plot(gw_range, gate_weights[:, 2], label='Image', marker='^', color='#F44336')
    axes[1, 1].set_title('Agent-Modulated Attention Weights')
    axes[1, 1].set_xlabel('Epochs'); axes[1, 1].set_ylabel('Softmax Weight')
    axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3); axes[1, 1].set_ylim(0, 1)
else:
    axes[1, 1].text(0.5, 0.5, 'No Weight History', ha='center', va='center', 
                   transform=axes[1, 1].transAxes, fontsize=14)

plt.tight_layout()
plt.show()

# Final Summary Table
print("\nModel Performance Summary:")
def safe_max(key, fallback):
    vals = history.get(key, [])
    return max(vals) if len(vals) > 0 else fallback

results_df = pd.DataFrame({
    'Metric': ['AUC', 'F1 Score', 'AP', 'Precision', 'Recall'],
    'Current': [cur_auc, cur_f1, cur_ap, cur_prec, cur_rec],
    'Best (History)': [
        safe_max('val_auc', cur_auc),
        safe_max('val_f1', cur_f1),
        safe_max('val_ap', cur_ap),
        cur_prec,  # Not tracked per-epoch
        cur_rec    # Not tracked per-epoch
    ]
})
print(results_df.to_string(index=False))